# 複数の戻り値では、4個以上の変数ならアンパックしない

アンパック構文の効果の 1つは、Python 関数が複数の値を返せることです。

例えば、ワニの生息地について様々な統計量を求めたいとします。体長のリストがあるので、母集団での最大と最小をけいさんします。2つの値を返す関数1つで行います。

In [1]:
def get_status(numbers):
  minimum = min(numbers)
  maximum = max(numbers)
  return minimum, maximum

lengths = [63, 73, 72, 60, 67, 66, 71, 61, 72, 70]

minimum, maximum = get_status(lengths)

print(f'Min: {minimum}, Max: {maximum}')

Min: 60, Max: 73


catch-all アンパックのアスタリスク付きの式でも複数の値を受け取れます。

例えば、ワニが母集団の平均に比べてどのくらい大きいか計算する関数を作ります。この関数は比率のリストを返しますが、リストの真ん中部分をアスタリスク付きの式で受けることにより、最大と最小を個別に受け取れます。

In [5]:
def get_avg_ratio(numbers):
  average = sum(numbers) / len(numbers)
  scaled = [x / average for x in numbers]
  scaled.sort(reverse=True)
  return scaled

longest, *middle, shortest = get_avg_ratio(lengths)

print(f'Longest:  {longest:>4.0%}')
print(f'Shortest: {shortest:>4.0%}')

Longest:  108%
Shortest:  89%


次に、プログラムへの要求が変更されたとします。

ワニの平均長、中央値、母集団のサイズも求めないといけません。これを、get_status 関数を拡張して、これらの統計量も計算して結果のタプルを呼び出し側でアンパックするようにします。

In [9]:
def get_status(numbers):
  minimum = min(numbers)
  maximum = max(numbers)
  count = len(numbers)
  average = sum(numbers) / count

  sorted_numbers = sorted(numbers)
  middle = count // 2
  if count % 2 == 0:
    lower = sorted_numbers[middle - 1]
    upper = sorted_numbers[middle]
    median = (lower + upper) / 2
  else:
    median = sorted_numbers[middle]

  return minimum, maximum, average, median, count

minimum, maximum, average, median, count = get_status(lengths)

print(f'Min: {minimum}, Max: {maximum}')
print(f'Average: {average}, Median: {median}, Count: {count}')

Min: 60, Max: 73
Average: 67.5, Median: 68.5, Count: 10


このコードには 2つの問題があります。

第1に、戻り値がすべて数値なので、順序を間違えやすく、後から特定しにくいバグの原因になりかねません。多数の戻り値を使うのは極めてエラーになりやすいものです。

第2に、関数を呼び出し、値をアンパックする行が長くなり、様々な方法で改良して続ける必要がありますが、そうすると読みにくくなってしまいます。

In [10]:
minimum, maximum, average, median, count = get_status(lengths)

minimum, maximum, average, median, count = \
  get_status(lengths)

(minimum, maximum, average,
 median, count) = get_status(lengths)

これらの問題を避けるには、関数からの複数の戻り値をアンパックするときに 4個以上の変数を決して使わないことです。
この場合は、3要素タプル、2変数と1つのアスタリスク付きの式、あるいはもっと短いものにします。4個以上の戻り値をアンパックする場合には、軽量クラスや namedtuple を定義し、関数でそのインスタンスを返すようにします。

## 覚えておくこと

- 関数で複数の値を返すには、それらをタプルに入れて、呼び出し側では Python のアンパック構文を使うことができる。
- 関数からの複数の戻り値は catch-all のアスタリスク付きの引数でアンパックすることもできる。
- 4個以上の変数をアンパックするのはエラーになりやすいので避けて、代わりに軽量なクラスか namedtuple インスタンスを使う。

## 補足

### 推奨例

辞書を返すパターン（シンプル）

In [11]:
def get_user_profile():
    return {
        "name": "Alice",
        "age": 28,
        "email": "alice@example.com",
        "address": "Tokyo",
        "phone": "090-1234-5678"
    }

profile = get_user_profile()
print(profile["email"])


alice@example.com


namedtuple を返すパターン（軽量＆順序もサポート）

In [12]:
from collections import namedtuple

Profile = namedtuple("Profile", ["name", "age", "email", "address", "phone"])

def get_user_profile():
    return Profile(
        name="Alice",
        age=28,
        email="alice@example.com",
        address="Tokyo",
        phone="090-1234-5678"
    )

profile = get_user_profile()
print(profile.email)


alice@example.com


dataclassを返すパターン（最近の標準）

In [ ]:
from dataclasses import dataclass

@dataclass
class Profile:
    name: str
    age: int
    email: str
    address: str
    phone: str

def get_user_profile():
    return Profile(
        name="Alice",
        age=28,
        email="alice@example.com",
        address="Tokyo",
        phone="090-1234-5678"
    )

profile = get_user_profile()
print(profile.email)

alice@example.com


どの方式を選べばいいか？

| 方式         | メリット                 | デメリット           | 適用例           |
| ---------- | -------------------- | --------------- | ------------- |
| 辞書         | 一番手軽。動的。柔軟。          | 型安全でない、typo に弱い | 小規模スクリプト      |
| namedtuple | 軽い、順序アクセスも可、フィールド名あり | 不変（書き換え不可）      | 返すだけで変更しないデータ |
| dataclass  | 拡張性◎、型サポート◎、人気       | 辞書より少し準備が必要     | 大きなコード・型管理あり  |


### detaclass の Private について

Python の「Private」は 厳密なアクセス制御ではなく、慣習ベースです。

基本的に Python の属性はすべて Public です。これは deataclass でも同じです。

「EAFP スタイルの疑似プライベート」：先頭に __ を付ける

In [19]:
from dataclasses import dataclass

@dataclass
class Profile:
    __password: str  # 擬似的に Private 的な扱い

p = Profile("secret")
print(p.__password)  # ❌ AttributeError

AttributeError: 'Profile' object has no attribute '__password'

代わりに名前が 名前マングリングされて内部的には _Profile__password になります

In [20]:
print(p._Profile__password)  # こうするとアクセスできてしまう

secret


正攻法：Private + Getter/Setter（プロパティ）

In [21]:
from dataclasses import dataclass, field

@dataclass
class Profile:
    __password: str = field(repr=False)  # repr にも表示させない

    @property
    def password(self):
        raise AttributeError("Password is write-only.")

    @password.setter
    def password(self, value):
        if len(value) < 8:
            raise ValueError("Password is too short.")
        self.__password = value

- 表示禁止
- 読み取り不可（例外を投げる）
- 書き込みは検証付きで可能